# importing libraries

In [1]:
!pip install gensim numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from gensim.models import KeyedVectors
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, f1_score
import gensim.downloader as api
from imblearn.over_sampling import ADASYN

# loading dataset

In [3]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# preprocessing

In [4]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.to_numpy(), X_test.to_numpy(), y_train.to_numpy(), y_test.to_numpy()

In [6]:
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# vocabulary stats

In [7]:
averageWordsLength = round(sum([len(i.split()) for i in df['text']]) / len(df['text']))
totalWordsLength = len(set(" ".join(df['text']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 4457
Average words per message: 16
Approximate vocabulary size: 15686


# more preprocessing

In [8]:
text_vec = tf.keras.layers.TextVectorization(
    max_tokens=totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=averageWordsLength,
)
text_vec.adapt(X_train)

X_train = text_vec(X_train)
X_test = text_vec(X_test)

In [9]:
vocab_size = text_vec.get_vocabulary()
word_index = dict(zip(vocab_size, range(len(vocab_size))))

In [10]:
ft = api.load("fasttext-wiki-news-subwords-300")

[==================================================] 100.0% 958.5/958.4MB downloaded


In [11]:
embedding_dim = 300
embedding_matrix = np.zeros((len(vocab_size), embedding_dim))

for word, i in word_index.items():
    if word in ft:
        embedding_matrix[i] = ft[word]

embedding_layer = tf.keras.layers.Embedding(
    input_dim=len(vocab_size),
    output_dim=embedding_dim,
    input_length=averageWordsLength,
    weights=[embedding_matrix],
    trainable=False
)

X_train = embedding_layer(X_train)
X_test = embedding_layer(X_test)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
train_counts = y_train.sum(axis=0)
test_counts = y_test.sum(axis=0)

print(f"Training set - Ham: {int(train_counts[0])}, Spam: {int(train_counts[1])}")
print(f"Testing set - Ham: {int(test_counts[0])}, Spam: {int(test_counts[1])}")

# num_classes = y_train.shape[1]
# print(f"\nTotal number of unique classes: {num_classes}")

Training set - Ham: 3859, Spam: 598
Testing set - Ham: 966, Spam: 149


In [13]:
# ADASYN expects 2D features (samples, features) and 1D labels.
# X_train is currently 3D (samples, sequence_length, embedding_dim)
# y_train is currently 2D (one-hot encoded)

n_samples, seq_len, embed_dim = X_train.shape
X_train_flat = np.reshape(X_train, (n_samples, seq_len * embed_dim))
y_train_1d = np.argmax(y_train, axis=1)

ada = ADASYN(random_state=42)
X_resampled, y_resampled = ada.fit_resample(X_train_flat, y_train_1d)

# Reshape X_train back to 3D and convert y_train back to one-hot encoding for the model
X_train = np.reshape(X_resampled, (X_resampled.shape[0], seq_len, embed_dim))
y_train = to_categorical(y_resampled, num_classes=2)

# Calculate and print the new class distribution
resampled_counts = y_train.sum(axis=0)
print(f"Training set after ADASYN - Ham: {int(resampled_counts[0])}, Spam: {int(resampled_counts[1])}")

Training set after ADASYN - Ham: 3859, Spam: 3907


# model development

In [14]:
cnn = tf.keras.models.Sequential()
cnn.add(tf.keras.layers.Input(shape=(12,300)))

In [15]:
cnn.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation="relu"))
cnn.add(tf.keras.layers.GlobalAveragePooling1D())
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn.add(tf.keras.layers.Dense(units=2, activation='softmax'))

In [16]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 8, 128)         │       192,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 208,898 (816.01 KB)

 Trainable params: 208,898 (816.01 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[tf.keras.metrics.F1Score(name='f1_score', average='weighted')]
)

# model training

In [18]:
history = cnn.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

Epoch 1/5
243/243 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - f1_score: 0.8666 - loss: 0.3311 - val_f1_score: 0.9422 - val_loss: 0.1637
Epoch 2/5
243/243 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - f1_score: 0.9721 - loss: 0.0792 - val_f1_score: 0.9496 - val_loss: 0.1656
Epoch 3/5
243/243 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - f1_score: 0.9865 - loss: 0.0466 - val_f1_score: 0.9549 - val_loss: 0.1481
Epoch 4/5
243/243 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - f1_score: 0.9904 - loss: 0.0319 - val_f1_score: 0.9613 - val_loss: 0.1358
Epoch 5/5
243/243 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - f1_score: 0.9982 - loss: 0.0110 - val_f1_score: 0.9663 - val_loss: 0.1698


# model evaluation

In [19]:
test_messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

test_messages_tf = tf.constant(test_messages, dtype=tf.string)
test_messages_tf = embedding_layer(text_vec(test_messages_tf))
preds = cnn.predict(test_messages_tf)

for msg, p in zip(test_messages, preds):
    label = "Spam" if np.argmax(p)==1 else "Ham"
    print(f"{label} → {msg} ({p[0]:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
Ham → Hey, are we meeting today? (1.00)
Ham → WIN cash now!!! Click the link (0.50)
Ham → Your OTP is 348921 (0.98)
Ham → You have won $1000! (0.54)
Spam → Congratulations! You have won a free lottery ticket (0.01)
Ham → Don't forget to submit the assignment (1.00)
Spam → URGENT! You won a cash prize. Call immediately! (0.00)
Spam → My friend won a prize yesterday (0.29)


In [20]:
print(classification_report(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))
print(f1_score(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       966
           1       0.91      0.84      0.87       149

    accuracy                           0.97      1115
   macro avg       0.94      0.91      0.93      1115
weighted avg       0.97      0.97      0.97      1115

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
0.8710801393728222
